In [ ]:
import fastf1
import fastf1.plotting
import pandas as pd
import numpy as np
import cv2
import os
import json
import subprocess
import ipywidgets as widgets
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont
from scipy.interpolate import interp1d, splprep, splev
from scipy.signal import correlate, medfilt
import warnings

warnings.simplefilter(action='ignore')

# ==========================================
# 0. DATABASE CONNECTION
# ==========================================
DB_DIR = r"C:\Users\91910\Downloads\openclaw-pipeline\database"
CALENDAR_DB = {}
DRIVERS_DB = {}
SEASON_DB = {}

try:
    with open(os.path.join(DB_DIR, "calendar_cache.json"), 'r', encoding='utf-8') as f:
        CALENDAR_DB = json.load(f)
    with open(os.path.join(DB_DIR, "drivers_by_race.json"), 'r', encoding='utf-8') as f:
        DRIVERS_DB = json.load(f)
    with open(os.path.join(DB_DIR, "drivers_by_season.json"), 'r', encoding='utf-8') as f:
        SEASON_DB = json.load(f)
    print(f"✅ Successfully linked local F1 database.")
except Exception as e:
    print(f"⚠️ Warning: Could not load local databases. Error: {e}")

# ==========================================
# 0b. F1 API PREFLIGHT CHECK
# ==========================================
def check_f1_api_access(timeout=15):
    """Quick probe of the F1 live-timing API. Returns (ok, message).
    Used to explain load failures such as a VPN/datacenter IP being blocked by F1's CDN."""
    import requests
    test_url = ("https://livetiming.formula1.com/static/2024/"
                "2024-03-02_Bahrain_Grand_Prix/2024-03-01_Qualifying/SessionInfo.json")
    try:
        r = requests.get(test_url, timeout=timeout)
    except Exception as e:
        return False, f"Could not reach the F1 API at all (network/DNS error: {e})."
    if r.status_code == 200:
        return True, "F1 API is reachable."
    if r.status_code == 403:
        try:
            ip = requests.get("https://api.ipify.org", timeout=8).text
        except Exception:
            ip = "unknown"
        return False, (
            f"F1 API returned 403 'Request blocked' (your outbound IP: {ip}). "
            "You are on a VPN / proxy / datacenter IP that F1's CDN blocks. "
            "FIX: disconnect your VPN and re-run. Already-cached races still work offline."
        )
    return False, f"F1 API returned unexpected status {r.status_code}."


# ==========================================
# 1-2. OPTIMAL x QUALI ENGINE
#   Delta engine = hifi_delta_v2.py (exact F1InsightsHub port); animation timing = the LAPCOMP.IPYNB mapping.
#   Road + racing line = the optimal lap built for this track (F1_Pipeline_Assets/tracks/<track>_lapcomp.json).
#   P1 drives the shipped optimal line; every other car gets its own optimiser line (~1% setup variation, cached).
# ==========================================
import sys, importlib
LAPCOMP_DIR = r"D:\f1_hot_lap\OPTIMAL LAP SCRIPTS\LAP COMPARISON"
if LAPCOMP_DIR not in sys.path: sys.path.insert(0, LAPCOMP_DIR)
import lap_compare_optimal as LCO, optimal_lines as OL
importlib.reload(LCO); importlib.reload(OL)
H = LCO.H

class F1VideoBaker:
    def __init__(self, track, configs, is_same_race=True, zoom_factor=1.0, trail_frames=60, fps=30):
        self.track = track
        self.configs = configs
        self.is_same_race = is_same_race
        self.fps = fps
        self.zoom_factor = zoom_factor
        self.trail_frames = trail_frames
        self.ref_config = configs[0]
        self.loaded_sessions = {}

    def get_session(self, year, session_type):
        key = (year, self.track, session_type)
        if key not in self.loaded_sessions:
            print(f"⏳ Loading Session: {year} {self.track} ({session_type})...")
            try: fastf1.Cache.enable_cache('cache')
            except: pass
            s = fastf1.get_session(year, self.track, session_type)
            s.load(telemetry=True, laps=True, weather=False, messages=False)
            self.loaded_sessions[key] = s
        return self.loaded_sessions[key]

    def get_clean_trace(self, config):
        try:
            session = self.get_session(config['year'], config['session'])
            laps = session.laps.pick_driver(config['driver'])
            if laps.empty: return None
            lap = laps.pick_fastest()
            tel = lap.get_telemetry().dropna(subset=['Distance', 'Speed', 'X', 'Y'])
            tel = tel.drop_duplicates(subset=['Time'])

            dist = tel['Distance'].values
            dist -= dist[0]

            return {
                'dist': dist,
                'speed': tel['Speed'].values / 3.6,
                'time': tel['Time'].dt.total_seconds().values,
                'x': tel['X'].values / 10.0,
                'y': tel['Y'].values / 10.0,
                'lap_time': lap['LapTime'].total_seconds(),
                'f1ih': H.get_trace(session, config['driver'], lap),   # delta engine input (same lap)
                'driver': config['driver'],
                'config': config
            }
        except Exception as e:
            print(f"⚠️ Error with {config['driver']} ({config['year']}): {e}")
            return None

    def format_display_name(self, config):
        if self.is_same_race: return config['driver']
        yr = str(config['year'])[-2:]
        return f"{config['driver']} '{yr} ({config['session']})"

    def bake(self):
        try:
            recipe = OL.load_recipe(self.track)
        except FileNotFoundError as e:
            print(f"❌ {e}"); return

        traces = []
        for cfg in self.configs:
            tr = self.get_clean_trace(cfg)
            if not tr:
                if cfg is self.ref_config:
                    print("❌ Reference driver trace missing.")
                    ok, msg = check_f1_api_access()
                    print(f"🚫 LIKELY CAUSE: {msg}" if not ok else "   (F1 API is reachable — verify the driver code / year / session are valid for this race.)")
                    return
                continue
            tr['label'] = self.format_display_name(cfg)
            traces.append(tr)
            print(f"   {tr['label']}: {tr['lap_time']:.3f}s")

        print(f"🛤️ Road + line from the optimal lap: {os.path.basename(recipe['raceline'])}")
        if len(traces) > 1:
            print(f"   making {len(traces) - 1} driver line(s) (first time per track: a few minutes each, then cached)")
        variants = OL.variant_lines(recipe, len(traces) - 1)
        line_paths = {traces[0]['label']: recipe['raceline']}
        for tr, p in zip(traces[1:], variants): line_paths[tr['label']] = p

        title1 = f"{self.track.upper()}" + (f" - {self.ref_config['year']}" if self.is_same_race else "")
        title2 = (f"TOP {len(traces)} - {self.ref_config['session'].upper()} LAP COMPARISON" if self.is_same_race
                  else "CROSS-SESSION LAP COMPARISON")
        out_filename = f"f1_{self.track.replace(' ', '_').lower()}_{self.ref_config['year']}_{self.ref_config['session'].lower()}_optimal_comparison.mp4"
        out_path = os.path.join(LAPCOMP_DIR, out_filename)
        print("📼 Rendering High-Speed MP4...")
        LCO.render(traces, recipe['outline'], line_paths, self.track, self.ref_config['year'],
                   self.ref_config['session'], out_path, zoom=float(recipe.get('zoom', 28)) * self.zoom_factor,
                   trail_frames=self.trail_frames, fps=self.fps, title1=title1, title2=title2,
                   log=lambda m: None if m.startswith("  frame") else print(m))
        print(f"✅ SUCCESS: Video saved as '{out_path}'")
        try: subprocess.run(f'explorer /select,"{out_path}"', shell=True)
        except: pass

# ==========================================
# 3. INTERACTIVE WIDGET UI PIPELINE
# ==========================================
years = sorted(list(CALENDAR_DB.keys()), reverse=True) if CALENDAR_DB else ['2024', '2023', '2022']
all_tracks = set()
for y, races in CALENDAR_DB.items():
    for r in races: all_tracks.add(r['event_name'])
all_tracks = sorted(list(all_tracks)) if all_tracks else ['Bahrain Grand Prix']

w_track = widgets.Dropdown(options=all_tracks, value=all_tracks[0], description='Track:')
w_same_race = widgets.Checkbox(value=True, description='Same Race (Lock Targets to Ref Year/Session)')

driver_rows = []
for i in range(5):
    y_drop = widgets.Dropdown(options=years, value=years[0], layout=widgets.Layout(width='80px'))
    s_drop = widgets.Dropdown(options=['Q', 'R', 'S', 'SQ', 'FP1', 'FP2', 'FP3'], value='Q', layout=widgets.Layout(width='60px'))
    d_drop = widgets.Dropdown(options=[], layout=widgets.Layout(width='120px'))
    driver_rows.append({'year': y_drop, 'session': s_drop, 'driver': d_drop})

def _extract_drivers_flat(node):
    if isinstance(node, list):
        for item in node:
            if isinstance(item, dict) and 'code' in item:
                yield item
    elif isinstance(node, dict):
        for v in node.values():
            yield from _extract_drivers_flat(v)

def update_driver_options(row_idx):
    row = driver_rows[row_idx]
    yr = row['year'].value
    trk = w_track.value
    d_list = []

    if str(yr) in DRIVERS_DB and trk in DRIVERS_DB[str(yr)]:
        seen = {}
        for d in _extract_drivers_flat(DRIVERS_DB[str(yr)][trk]):
            if d['code'] not in seen:
                seen[d['code']] = True
        d_list = list(seen.keys())

    if not d_list and str(yr) in SEASON_DB:
        season = SEASON_DB[str(yr)]
        d_list = [d['code'] for d in _extract_drivers_flat(season)]
        
    if not d_list:
        d_list = ['VER', 'PER', 'LEC', 'SAI', 'NOR', 'PIA', 'HAM', 'RUS', 'ALO', 'STR']
        
    if row_idx > 0:
        d_list = [""] + d_list
        
    old_val = row['driver'].value
    row['driver'].options = d_list
    if old_val in d_list:
        row['driver'].value = old_val

def on_global_change(*args):
    for i in range(len(driver_rows)): update_driver_options(i)

w_track.observe(on_global_change, 'value')
for i, r in enumerate(driver_rows):
    r['year'].observe(lambda change, idx=i: update_driver_options(idx), 'value')

def sync_same_race(*args):
    is_locked = w_same_race.value
    ref_y = driver_rows[0]['year'].value
    ref_s = driver_rows[0]['session'].value
    for i in range(1, 5):
        driver_rows[i]['year'].disabled = is_locked
        driver_rows[i]['session'].disabled = is_locked
        if is_locked:
            driver_rows[i]['year'].value = ref_y
            driver_rows[i]['session'].value = ref_s

w_same_race.observe(sync_same_race, 'value')
driver_rows[0]['year'].observe(sync_same_race, 'value')
driver_rows[0]['session'].observe(sync_same_race, 'value')

on_global_change()
sync_same_race()

w_zoom = widgets.FloatSlider(value=1.0, min=0.25, max=3.0, step=0.05, description='Zoom x:')  # x the optimal-lap zoom
w_trail = widgets.IntSlider(value=60, min=0, max=300, step=10, description='Trail Frames:')

btn_gen = widgets.Button(description="GENERATE MANUAL", button_style='success')
btn_top2 = widgets.Button(description="AUTO TOP 2", button_style='warning')
btn_top3 = widgets.Button(description="AUTO TOP 3", button_style='info')
btn_top5 = widgets.Button(description="AUTO TOP 5", button_style='primary')
out = widgets.Output()

def get_auto_top_drivers(n):
    year = int(driver_rows[0]['year'].value)
    track = w_track.value
    session_type = driver_rows[0]['session'].value
    
    print(f"🔍 Auto-detecting Top {n} drivers for {year} {track} ({session_type})...")
    try: fastf1.Cache.enable_cache('cache')
    except: pass
    
    s = fastf1.get_session(year, track, session_type)
    s.load(telemetry=False, weather=False, messages=False)
    
    if s.laps.empty:
        print("❌ Could not find valid laps for this session.")
        ok, msg = check_f1_api_access()
        if not ok:
            print(f"🚫 LIKELY CAUSE: {msg}")
        return []
        
    fastest_laps = s.laps.pick_quicklaps().sort_values(by='LapTime')
    seen = []
    for _, lap in fastest_laps.iterrows():
        drv = lap['Driver']
        if drv not in seen: seen.append(drv)
        if len(seen) == n: break
        
    return seen

def on_auto_generate(n):
    with out:
        out.clear_output()
        seen_drivers = get_auto_top_drivers(n)
        if not seen_drivers: return
            
        print(f"🏁 Found Top {n}: {', '.join(seen_drivers)}")
        
        w_same_race.value = True
        for i, r in enumerate(driver_rows):
            if i < len(seen_drivers):
                d = seen_drivers[i]
                if d not in r['driver'].options:
                    r['driver'].options = list(r['driver'].options) + [d]
                r['driver'].value = d
            else:
                if "" not in r['driver'].options:
                    r['driver'].options = [""] + list(r['driver'].options)
                r['driver'].value = ""
        
        configs = [{'driver': d, 'year': int(driver_rows[0]['year'].value), 'session': driver_rows[0]['session'].value} for d in seen_drivers]
        
        baker = F1VideoBaker(
            track=w_track.value,
            configs=configs,
            is_same_race=True,
            zoom_factor=w_zoom.value,
            trail_frames=w_trail.value
        )
        baker.bake()

def on_manual_generate(b):
    with out:
        out.clear_output()
        configs = []
        for i, r in enumerate(driver_rows):
            d_val = r['driver'].value
            if d_val and str(d_val).strip() != '':
                configs.append({
                    'driver': d_val,
                    'year': int(r['year'].value),
                    'session': r['session'].value
                })
                
        if not configs:
            print("❌ Select at least a reference driver.")
            return
            
        baker = F1VideoBaker(
            track=w_track.value,
            configs=configs,
            is_same_race=w_same_race.value,
            zoom_factor=w_zoom.value,
            trail_frames=w_trail.value
        )
        baker.bake()

btn_gen.on_click(on_manual_generate)
btn_top2.on_click(lambda b: on_auto_generate(2))
btn_top3.on_click(lambda b: on_auto_generate(3))
btn_top5.on_click(lambda b: on_auto_generate(5))

ui_rows = [widgets.HBox([widgets.Label(f"{'Ref' if i==0 else f'Tgt {i}'}:", layout=widgets.Layout(width='40px')), r['year'], r['session'], r['driver']]) for i, r in enumerate(driver_rows)]
buttons_box = widgets.HBox([btn_gen, btn_top2, btn_top3, btn_top5])

ui = widgets.VBox([
    widgets.HTML("<h2>🏎️ OPTIMAL x QUALI — Lap Comparison on the Optimal-Lap Road</h2>"),
    w_track, w_same_race,
    widgets.HTML("<hr><b>Driver Roster:</b>"),
    *ui_rows,
    widgets.HTML("<hr><b>Visual Settings:</b>"),
    widgets.HBox([w_zoom, w_trail]),
    widgets.HTML("<hr>"),
    buttons_box,
    out
])

display(ui)